# LAB 09 - Tokenization và label alignment


In [ ]:
from collections import Counter
import pandas as pd
from baseline_ner import ROOT

def tokenization_analysis(rows,tokenizer):
    records=[]
    for row in rows[:max(20,min(100,len(rows)))]:
        e=tokenizer(row['tokens'],is_split_into_words=True)
        counts=Counter(w for w in e.word_ids() if w is not None)
        record={'id':row['id'],'words':len(row['tokens']),'tokens':sum(counts.values()),
                'split_words':sum(v>1 for v in counts.values()),'extra_subwords':sum(v-1 for v in counts.values()),
                'R':sum(counts.values())/len(row['tokens'])}
        for group in ['entity','non_entity']:
            indices=[i for i,t in enumerate(row['tags']) if (t!='O')==(group=='entity')]
            record[group+'_words']=len(indices); record[group+'_tokens']=sum(counts[i] for i in indices)
        records.append(record)
    frame=pd.DataFrame(records); frame.to_csv(ROOT/'training_output/tokenization_analysis.csv',index=False)
    return frame


In [1]:
from pathlib import Path
from transformers import AutoTokenizer
from baseline_ner import ROOT, MODEL, LABELS, load_data, align
import pandas as pd
Path('training_output').mkdir(exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True, model_max_length=512)
data = load_data()

F:\HUS\.venv-ner\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Word IDs và nhãn sau alignment


In [2]:
words = ['Elon', 'Musk', 'founded', 'SpaceX', 'in', 'California', '.']
tags = ['B-PER', 'I-PER', 'O', 'B-ORG', 'O', 'B-LOC', 'O']
encoded = align(tokenizer, words, tags)
display(pd.DataFrame({'token':tokenizer.convert_ids_to_tokens(encoded['input_ids']), 'input_ids':encoded['input_ids'], 'attention_mask':encoded['attention_mask'], 'word_ids':encoded.word_ids(), 'labels':encoded['labels'], 'BIO':[LABELS[i] if i != -100 else 'IGNORE' for i in encoded['labels']]}))
assert sum(i != -100 for i in encoded['labels']) == len(words)

,token,input_ids,attention_mask,word_ids,labels,BIO
0,[CLS],101,1,NaN,-100,IGNORE
1,el,3449,1,0.0,1,B-PER
2,##on,2239,1,0.0,-100,IGNORE
3,mu,14163,1,1.0,2,I-PER
4,##sk,6711,1,1.0,-100,IGNORE
5,founded,2631,1,2.0,0,O
6,space,2686,1,3.0,3,B-ORG
7,##x,2595,1,3.0,-100,IGNORE
8,in,1999,1,4.0,0,O
9,california,2662,1,5.0,5,B-LOC


## Loss và logits


In [3]:
import torch
torch.set_num_threads(2)
from transformers import DataCollatorForTokenClassification, AutoModelForTokenClassification
batch = DataCollatorForTokenClassification(tokenizer)([dict(encoded), dict(align(tokenizer, ['Paris', '.'], ['B-LOC', 'O']))])
model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=9, id2label=dict(enumerate(LABELS)), label2id={t:i for i,t in enumerate(LABELS)}, use_safetensors=True)
model.eval()
with torch.no_grad():
    output = model(**batch, output_hidden_states=True)
print('labels:', batch['labels'])
print('hidden states:', output.hidden_states[-1].shape)
print('logits:', output.logits.shape)
print('predictions (head chưa fine-tune):', output.logits.argmax(-1))
print('loss:', output.loss.item())

Some weights of BertForTokenClassification were not initialized from the model checkpoint at google/bert_uncased_L-2_H-128_A-2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


labels: tensor([[-100,    1, -100,    2, -100,    0,    3, -100,    0,    5,    0, -100],
        [-100,    5,    0, -100, -100, -100, -100, -100, -100, -100, -100, -100]])
hidden states: torch.Size([2, 12, 128])
logits: torch.Size([2, 12, 9])
predictions (head chưa fine-tune): tensor([[1, 2, 2, 2, 2, 6, 6, 1, 2, 1, 2, 2],
        [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]])
loss: 2.16483211517334


## Thí nghiệm trên 100 câu test


In [4]:
frame = tokenization_analysis(data['test'], tokenizer)
display(frame)
print('Số câu:', len(frame), 'R tổng:', frame.tokens.sum()/frame.words.sum())
frame['length_group'] = frame.words.le(frame.words.median()).map({True:'short', False:'long'})
summary = frame.groupby('length_group')[['words','tokens','split_words']].sum()
summary['R'] = summary.tokens/summary.words
display(summary)
for group in ['entity','non_entity']:
    print(group, 'R =', frame[group+'_tokens'].sum()/frame[group+'_words'].sum())

,id,words,tokens,split_words,extra_subwords,R,entity_words,entity_tokens,non_entity_words,non_entity_tokens
0,0,12,12,0,0,1.00,2,2,10,10
1,1,2,4,2,2,2.00,2,4,0,0
2,2,6,12,2,6,2.00,4,6,2,6
3,3,25,27,1,2,1.08,4,4,21,23
4,4,25,27,1,2,1.08,2,2,23,25
...,...,...,...,...,...,...,...,...,...,...
95,95,8,10,1,2,1.25,3,3,5,7
96,96,3,3,0,0,1.00,0,0,3,3
97,97,1,1,0,0,1.00,1,1,0,0
98,98,10,12,2,2,1.20,2,3,8,9


Số câu: 100 R tổng: 1.3037974683544304


,words,tokens,split_words,R
length_group,,,,
long,1127,1378,163,1.222715
short,295,476,99,1.613559


entity R = 1.8035714285714286
non_entity R = 1.149171270718232


In [5]:
from collections import Counter
split_examples = Counter()
for row in data['test'][:100]:
    enc = tokenizer(row['tokens'], is_split_into_words=True)
    counts = Counter(w for w in enc.word_ids() if w is not None)
    split_examples.update(row['tokens'][i] for i,n in counts.items() if n>1)
display(pd.DataFrame(split_examples.most_common(20), columns=['split_word','occurrences']))

,split_word,occurrences
0,Bitar,9
1,1996-12-06,6
2,Takagi,5
3,'s,5
4,Cuttitta,5
5,U.S.,5
6,Takuya,4
7,Nader,4
8,Jokhadar,4
9,Coste,4
